In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import os
import gc
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
import tqdm as notebook_tqdm

from tqdm.auto import tqdm
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 

from utils import reduce_mem_usage, plot_confusion_matrix

In [2]:
path_raw = os.path.join(os.path.abspath(os.curdir), 'raw_data')
path_prep = os.path.join(os.path.abspath(os.curdir), 'prep_data')
path_int8 = os.path.join(os.path.abspath(os.curdir), 'prep_int8_data')

In [3]:
num_parts_total = len([filename for filename in os.listdir(path_raw) if filename.startswith('train')])

In [4]:
def read_parquet_dataset_from_local(path_to_dataset: str, path_to_save: str, start_from: int = 0,
                                    num_parts_to_read: int = 1, columns: list=None, verbose: bool=False, 
                                    del_duplicates: bool=False, del_isna: bool=False) -> dict:
    """
    Функция:
    1. читает num_parts_to_read партиций, пременяет к ним функцию reduce_mem_usage для уменьшения размера файла в оперативной памяти,
    удаляет дубликаты (опционально del_duplicates=True) и пропуски (опционально del_isna=True), 
    записывает преобразованный файл в path_to_save
    2. проходясь по партициям собирает словарь df_dict уникальных значений по каждой колонке для последующего использования 
    в OneHotEncoder
    3. возвращает словарь df_dict
    
    :param path_to_dataset: str
        путь до директории с партициями
    :param path_to_save: str
        путь до директории в которую будут сохранены преобразованные партиции
    :param start_from: int
        номер партиции, с которой нужно начать чтение
    :param num_parts_to_read: int
        количество партиций, которые требуется прочитать
    :param columns: list
        список колонок, которые нужно прочитать из партиции
    :param verbose: bool
        флаг, выводить ли подробную информацию о выполнении
    :param del_duplicates: bool
        флаг, удалять ли дубликаты в обработанной партиции
    :param del_isna: bool
        флаг, удалять ли пропуски в обработанной партиции
        
    :return: dict
        словарь признак - уникальные значения по обработанным партициям
    """

    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)
                              if filename.startswith('train')])

    start_from = max(0, start_from)
    chunks = dataset_paths[start_from: start_from + num_parts_to_read]
    if verbose:
        print('Читаем часть данных:\n')
        for chunk in chunks:
            print(chunk)
    
    df_dict = None

    for chunk_path in notebook_tqdm.tqdm_notebook(chunks, desc='Читаем датасет с помощью Pandas'):
        if verbose:
            print('='*100)
            print('Путь к части данных', chunk_path)
        chunk = pd.read_parquet(chunk_path,columns=columns)
        if verbose:
            print(f'Кол-во записей в партиции: {chunk.shape[0]}')
            print(f'Размер в оперативной памяти: {chunk.memory_usage().sum() / 1024**3:0.2f} Gb')
        chunk = reduce_mem_usage(chunk)
        if verbose:
            print(f'Кол-во дубликатов: {(100 * chunk.duplicated().sum()/len(chunk)):4.2f}%')
        if del_duplicates:
            if verbose:
                print(f'До удаления дубликатов: размер чанка - {chunk.shape}, количество дубликатов - {chunk.duplicated().sum()}')
            chunk = chunk.drop_duplicates()
            if verbose:
                print(f'После удаления дубликатов: размер чанка - {chunk.shape}, количество дубликатов - {chunk.duplicated().sum()}')
        if verbose:
            print(f'Кол-во пропусков: {chunk.isna().sum().sum()}')
        if del_isna and chunk.isna().sum().sum() > 0:
            chunk = chunk.df.dropna()
            if verbose:
                print('Пропуски удалены')
        save_path = chunk_path.replace(path_to_dataset, path_to_save).replace('.pq', '_int8.pq')
        chunk.to_parquet(save_path, engine='pyarrow')
        if verbose:
            print('Преобразованная партиция сохранена')
        if df_dict is None:
            df_dict = {}
            for column in chunk.columns:
                df_dict[column] = set(chunk[column].unique())
            else:
                for column in chunk.columns:
                    unique = set(chunk[column].unique()) | df_dict[column]
                    df_dict[column] = unique
    
    for key in list(df_dict.keys()):
        df_dict[key] = list(df_dict[key])
    
    del chunk
    gc.collect()
        
    return df_dict

In [23]:
def prepare_dataset(path_to_dataset: str, ohe_dict: dict=None, save_to_path: str=None, verbose: bool=False) -> bool:
    
    """
    Выполняет предобработку партиций:
    1. кодирует с помощью OneHotEncoder
    2. агрегирует полученный датасет по id (не индекс!) клиента
    3. возвращает флаг успешного завершения True
    
    :param path_to_dataset: str
        путь до партиций
    :param ohe_dict: dict
        словарь для передачи в параметр categories OneHotEncoder с парами ключ - названия признаков,
        значения - уникальные значения признаков
    :param save_to_path: str
        путь до папки, в которой будет сохранён каждый обработанный блок в .parquet-формате; если None, то не будет сохранён
    :param verbose: bool
        флаг, выводить ли подробную информацию о выполнении
        
    :return: bool
        флаг успешного завершения
    """
    ohe_dict.pop('id', None)
    
    columns_to_encode = list(ohe_dict.keys())
    
    if ohe_dict is None:
        ohe = OneHotEncoder(
            sparse_output=False, 
            handle_unknown='ignore', 
            drop='first'
        )
    else:
        ohe = OneHotEncoder(
            categories=[df_dict[key] for key in columns_to_encode], 
            sparse_output=False, 
            handle_unknown='ignore', 
            drop='first'
        )
    
    dataset_paths = sorted([os.path.join(path_to_dataset, filename) for filename in os.listdir(path_to_dataset)])
    
    
    for step in notebook_tqdm.tqdm_notebook(range(0, len(dataset_paths)), desc="Преобразование данных parquet"):

        transactions_frame = pd.read_parquet(dataset_paths[step])
        if verbose:
            print(f'Размер датасета в оперативной памяти: {transactions_frame.memory_usage().sum()/1024**3:.3f} Gb')

   # Препроцессинг данных
        
        # - кодируем данные
        
        if step == 0:
            encoded_frame = ohe.fit_transform(transactions_frame[columns_to_encode])
            first_file = False
        else:
            encoded_frame = ohe.transform(transactions_frame[columns_to_encode])
        
        encoded_frame = pd.DataFrame(data=encoded_frame, columns=ohe.get_feature_names_out()).astype('int8')
        encoded_frame = pd.concat([transactions_frame, encoded_frame], axis=1)
        
        del transactions_frame
        gc.collect()
        
        data_preprocessed = encoded_frame.drop(columns=columns_to_encode)
        
        del encoded_frame
        gc.collect()
                
        # - агрегируем данные по id
        
        data_preprocessed = data_preprocessed.groupby('id', as_index=False).agg('sum')
        

    # Записываем подготовленные данные в файл
                

        if save_to_path == None:
            save_to_path = os.path.join(os.path.abspath(os.curdir), 'prep_data')
            
        
        if not os.path.exists(save_to_path):
            os.makedirs(save_to_path)
            if verbose:
                print('!'*100)
                print(f'Директория {save_to_path} создана')
                print('!'*100)

            
        block_as_str = str(step)
        if len(block_as_str) == 1:
            block_as_str = '00' + block_as_str
        else:
            block_as_str = '0' + block_as_str
                
        data_preprocessed.to_parquet(os.path.join(save_to_path, f'processed_chunk_{block_as_str}.parquet'))
        
        if verbose:
            print(f'Количество записей в обработанном файле: {data_preprocessed.shape[0]}')
            print(f'В файле содержатся данные по клиентам с id от {data_preprocessed["id"].min()} до {data_preprocessed["id"].max()}')
            print(f'Размер датасета в оперативной памяти {data_preprocessed.memory_usage().sum()/1024**3:.3f} Gb')
        del data_preprocessed
        gc.collect()
        if verbose:
            print('!'*100)
            print(f'Предобработанная партиция записана в файл "processed_chunk_{block_as_str}.parquet"')
            print('!'*100)

    if verbose:
        print('Подготовка промежуточных партиций произведена')
    
    return True

In [24]:
def prepare_full_dataset(path_to_prepared_datasets: str, path_to_targets: str, verbose: bool=False) -> pd.DataFrame:

    """
    Выполняет:
    1. обединение партиций в pd.DataFrame
    2. соединяет партиции по 'id' клиента с целевой меткой 'flag'
    
    :param path_to_dataset: str
        путь до партиций
    :path_to_targets: str
        путь до датасета с целевыми метками
    :param verbose: bool
        флаг, выводить ли подробную информацию о выполнении

    :return: pd.DataFrame
        результирующий датафрейм с целевыми метками
    """
    
    preprocessed_flag = 0


    dataset_paths = sorted([os.path.join(path_to_prepared_datasets, filename) for filename in os.listdir(path_to_prepared_datasets)
                              if filename.startswith('processed')])

    for dataset_path in notebook_tqdm.tqdm_notebook(dataset_paths, desc="Подготовливаем датасет с помощью Pandas"):
        print('Часть', dataset_path)
        chunk = pd.read_parquet(dataset_path)
        if preprocessed_flag == 0:
            result_dataset = chunk.copy()
            print(f'Инициализирован датасет. прочитано {result_dataset.shape[0]} строк. Количество столбцов {result_dataset.shape[1]}')
            preprocessed_flag = 1
        else:
            result_dataset = pd.concat([result_dataset, chunk], axis=0, ignore_index=True)
            
            print(f'Прочитано {result_dataset.shape[0]} строк. Количество столбцов {result_dataset.shape[1]}')
    
    print(f'Количество отсутствующих значений после конкатенации: {result_dataset.isna().sum().sum()}')
    result_dataset.fillna(0, inplace=True)
    print(f'Количество отсутствующих значений после обработки пропусков: {result_dataset.isna().sum().sum()}')
    print(f'Обрабатываем целевые переменные')
    targets = pd.read_csv(os.path.join(path_to_targets, 'target.csv'))
    result_dataset = result_dataset.merge(targets, how='inner', on='id')
    result_dataset[result_dataset.columns[1:]] = result_dataset[result_dataset.columns[1:]].astype('int8')
    if verbose:
        print(f'Размерность полученного датасета: {result_dataset.shape}, количество пропусков {result_dataset.isna().sum().sum()}')
        print(f'Размер полученного датасета в оперативной памяти {result_dataset.memory_usage().sum()/1024**3:.3f} Gb')

    return result_dataset

In [30]:
def prepare_scaled_files(columns: list, path_df_train: str=None, path_df_test: str=None, save_to_path: str=None, 
                         step: int=-1, verbose: bool=False) -> bool:
    
    
    columns_to_transform = [column for column in columns if column not in ['id', 'flag', '__index_level_0__']]
    columns = columns_to_transform.copy()
    columns.append('id')
    std_columns = [column + '_std' for column in columns_to_transform]
    
    scaler_std = StandardScaler()
    
    if path_df_train == None:
        path_df_train = os.path.join(os.path.abspath(os.curdir), 'data_train.pq')
    if path_df_test == None:
        path_df_test = os.path.join(os.path.abspath(os.curdir), 'data_test.pq')
    if save_to_path == None:
        save_to_path = os.path.join(os.path.abspath(os.curdir), 'std_data')
    if step < 0:
        train_name = os.path.join(save_to_path, 'df_train_std.parquet')
        test_name = os.path.join(save_to_path, 'df_test_std.parquet')
    else:
        train_name = os.path.join(save_to_path, f'df_train_std_{step}.parquet')
        test_name = os.path.join(save_to_path, f'df_test_std_{step}.parquet')
    if not os.path.exists(save_to_path):
        os.makedirs(save_to_path)
        if verbose:
            print('!'*100)
            print(f'Директория {save_to_path} создана')
            print('!'*100)
        
    df_train = pd.read_parquet(path_df_train, columns=columns)
    if verbose:
        print(f'Размерность загруженного df_train: {df_train.shape}')
        print(f'Размер загруженного df_train в оперативной памяти {df_train.memory_usage().sum()/1024**3:.3f} Gb')
    df_train[std_columns] = scaler_std.fit_transform(df_train[columns_to_transform])
    df_train.drop(columns=columns_to_transform, inplace=True)
    df_train.to_parquet(train_name)
    if verbose:
        print(f'Размерность df_train после маштабирования df_train: {df_train.shape}')
        print(f'Размер df_train в оперативной памяти после маштабирования {df_train.memory_usage().sum()/1024**3:.3f} Gb')
        print(f'Файл {train_name} сохранен')
    del df_train
    gc.collect()

    df_test = pd.read_parquet(path_df_test, columns=columns)
    if verbose:
        print(f'Размерность загруженного датасета df_test: {df_test.shape}')
        print(f'Размер загруженного df_test в оперативной памяти {df_test.memory_usage().sum()/1024**3:.3f} Gb')
    df_test[std_columns] = scaler_std.transform(df_test[columns_to_transform])
    df_test.drop(columns=columns_to_transform, inplace=True)
    df_test.to_parquet(test_name)
    if verbose:
        print(f'Размерность загруженного df_test после маштабирования: {df_test.shape}')
        print(f'Размер df_test в оперативной памяти после маштабирования {df_test.memory_usage().sum()/1024**3:.3f} Gb')
        print(f'Файл {test_name} сохранен')
    del df_test
    gc.collect()
    return True

In [7]:
df_dict = read_parquet_dataset_from_local(path_raw, path_int8, num_parts_to_read=num_parts_total, verbose=True, del_duplicates=True)

Читаем часть данных:

/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_0.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_1.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_10.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_11.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_2.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_3.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_4.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_5.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_6.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_7.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_8.pq
/home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_9.pq


Читаем датасет с помощью Pandas:   0%|          | 0/12 [00:00<?, ?it/s]

Путь к части данных /home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_0.pq
Кол-во записей в партиции: 1974724
Размер в оперативной памяти: 0.90 Gb
Используемая память до преобразования 0.897 Gb


  0%|          | 0/61 [00:00<?, ?it/s]

Используемая память после преобразования: 0.118 Gb
Количество используемой памяти уменьшено на 86.9%
Кол-во дубликатов: 0.00%
До удаления дубликатов: размер чанка - (1974724, 61), количество дубликатов - 0
После удаления дубликатов: размер чанка - (1974724, 61), количество дубликатов - 0
Кол-во пропусков: 0
Преобразованная партиция сохранена
Путь к части данных /home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_1.pq
Кол-во записей в партиции: 2107305
Размер в оперативной памяти: 0.96 Gb
Используемая память до преобразования 0.958 Gb


...

Используемая память после преобразования: 0.132 Gb
Количество используемой памяти уменьшено на 86.9%
Кол-во дубликатов: 0.00%
До удаления дубликатов: размер чанка - (2222245, 61), количество дубликатов - 0
После удаления дубликатов: размер чанка - (2222245, 61), количество дубликатов - 0
Кол-во пропусков: 0
Преобразованная партиция сохранена
Путь к части данных /home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_8.pq
Кол-во записей в партиции: 2242615
Размер в оперативной памяти: 1.02 Gb
Используемая память до преобразования 1.019 Gb


  0%|          | 0/61 [00:00<?, ?it/s]

Используемая память после преобразования: 0.134 Gb
Количество используемой памяти уменьшено на 86.9%
Кол-во дубликатов: 0.00%
До удаления дубликатов: размер чанка - (2242615, 61), количество дубликатов - 0
После удаления дубликатов: размер чанка - (2242615, 61), количество дубликатов - 0
Кол-во пропусков: 0
Преобразованная партиция сохранена
Путь к части данных /home/vadim/Skillbox/ml-junior/final_work/raw_data/train_data_9.pq
Кол-во записей в партиции: 2284256
Размер в оперативной памяти: 1.04 Gb
Используемая память до преобразования 1.038 Gb


  0%|          | 0/61 [00:00<?, ?it/s]

Используемая память после преобразования: 0.136 Gb
Количество используемой памяти уменьшено на 86.9%
Кол-во дубликатов: 0.00%
До удаления дубликатов: размер чанка - (2284256, 61), количество дубликатов - 0
После удаления дубликатов: размер чанка - (2284256, 61), количество дубликатов - 0
Кол-во пропусков: 0
Преобразованная партиция сохранена


In [8]:
prepare_dataset(path_int8, ohe_dict=df_dict, save_to_path=None, verbose=True)

Преобразование данных parquet:   0%|          | 0/12 [00:00<?, ?it/s]

Размер датасета в оперативной памяти: 0.118 Gb
Количество записей в обработанном файле: 250000
В файле содержатся данные по клиентам с id от 0 до 249999
Размер датасета в оперативной памяти 0.091 Gb
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Предобработанная партиция записана в файл "processed_chunk_000.parquet"
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Размер датасета в оперативной памяти: 0.137 Gb
Количество записей в обработанном файле: 250000
В файле содержатся данные по клиентам с id от 2500000 до 2749999
Размер датасета в оперативной памяти 0.091 Gb
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Предобработанная партиция записана в файл "processed_chunk_001.parquet"
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Размер датасета в оперативной памяти: 0.146 Gb


True

In [9]:
result_dataset = prepare_full_dataset(path_prep, path_raw, verbose=True)

Подготовливаем датасет с помощью Pandas:   0%|          | 0/12 [00:00<?, ?it/s]

Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_000.parquet
Инициализирован датасет. прочитано 250000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_001.parquet
Прочитано 500000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_002.parquet
Прочитано 750000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_003.parquet
Прочитано 1000000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_004.parquet
Прочитано 1250000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_005.parquet
Прочитано 1500000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-junior/final_work/prep_data/processed_chunk_006.parquet
Прочитано 1750000 строк. Количество столбцов 387
Часть /home/vadim/Skillbox/ml-jun

In [10]:
result_dataset['flag'].value_counts()

flag
0    2893558
1     106442
Name: count, dtype: int64

In [11]:
data_train, data_test = train_test_split(result_dataset, train_size=0.8, random_state=42, stratify=result_dataset['flag'])

In [12]:
data_train['flag'].value_counts()

flag
0    2314846
1      85154
Name: count, dtype: int64

In [13]:
data_test['flag'].value_counts()

flag
0    578712
1     21288
Name: count, dtype: int64

In [19]:
data_train.to_parquet('data_train.pq')

In [20]:
data_test.to_parquet('data_test.pq')

In [21]:
result_dataset.to_parquet('result_dataset.pq')

In [ ]:
prepare_scaled_files(verbose=True)

In [31]:
schema = pq.read_schema('data_train.pq')
column_names = schema.names

In [33]:
step = 50
num_interval = 0
test = []
for i in range(0, len(column_names), step):
    col_names = column_names[i:i+step].copy()
    print('V'*50)
    print(f'Читаем колонки {columns}')
    prepare_scaled_files(columns=col_names, step=num_interval, verbose=True)
    num_interval += 1

VVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVVV
Читаем колонки ['enc_loans_credit_type_4', 'enc_loans_credit_type_5', 'enc_loans_account_cur_1', 'enc_loans_account_cur_2', 'enc_loans_account_cur_3', 'pclose_flag_1', 'fclose_flag_1', 'flag', '__index_level_0__']
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Директория /home/vadim/Skillbox/ml-junior/final_work/std_data создана
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Размерность загруженного df_train: (2400000, 50)
Размер загруженного df_train в оперативной памяти 0.136 Gb
Размерность df_train после маштабирования df_train: (2400000, 50)
Размер df_train в оперативной памяти после маштабирования 0.903 Gb
Файл /home/vadim/Skillbox/ml-junior/final_work/std_data/df_train_std_0.parquet сохранен
Размерность загруженного датасета df_test: (600000, 50)
Размер загруженного df_test в оперативной памяти 0.034 Gb
Размерность загружен

In [36]:
data = pd.read_parquet(r'std_data/df_train_std_1.parquet')

In [37]:
data.head()

,id,rn_51_std,pre_since_opened_1_std,pre_since_opened_2_std,pre_since_opened_3_std,pre_since_opened_4_std,pre_since_opened_5_std,pre_since_opened_6_std,pre_since_opened_7_std,pre_since_opened_8_std,...,pre_pterm_4_std,pre_pterm_5_std,pre_pterm_6_std,pre_pterm_7_std,pre_pterm_8_std,pre_pterm_9_std,pre_pterm_10_std,pre_pterm_11_std,pre_pterm_12_std,pre_pterm_13_std
2305221,1805221,-0.001443,2.057688,0.747235,-0.543440,0.752423,-0.532813,-0.575035,0.727767,2.633210,...,0.834935,-0.016533,-0.56276,0.622118,-0.516139,0.626632,-0.315946,-0.593333,-0.539156,1.921067
924193,424193,-0.001443,-0.569485,-0.564254,-0.543440,-0.541460,0.802511,1.689727,-0.562101,-0.488118,...,-0.267581,-0.016533,-0.56276,1.782369,4.212771,0.626632,-0.315946,0.650515,0.802414,-0.536868
1396388,896388,-0.001443,0.744102,0.747235,2.149696,-0.541460,-0.532813,1.689727,-0.562101,-0.488118,...,1.386193,-0.016533,-0.56276,0.622118,-0.516139,0.626632,-0.315946,1.894364,-0.539156,0.692099
342966,2592966,-0.001443,-0.569485,-0.564254,-0.543440,-0.541460,-0.532813,-0.575035,-0.562101,-0.488118,...,-0.267581,-0.016533,-0.56276,-0.538133,-0.516139,-0.569912,-0.315946,-0.593333,-0.539156,-0.536868
2813279,2313279,-0.001443,-0.569485,-0.564254,-0.543440,-0.541460,-0.532813,-0.575035,-0.562101,2.633210,...,0.834935,-0.016533,-0.56276,-0.538133,-0.516139,0.626632,2.258718,0.650515,0.802414,3.150035


In [15]:
std_dir = os.path.join(os.path.abspath(os.curdir), 'std_data')
std_dir

'/home/vadim/Skillbox/ml-junior/final_work/std_data'

In [14]:
train_files = [file for file in os.listdir(std_dir) if file.startswith('df_train')]
train_files

['df_train_std_2.parquet',
 'df_train_std_6.parquet',
 'df_train_std_3.parquet',
 'df_train_std_0.parquet',
 'df_train_std_7.parquet',
 'df_train_std_5.parquet',
 'df_train_std_1.parquet',
 'df_train_std_4.parquet']

In [16]:
test_files = [file for file in os.listdir(std_dir) if file.startswith('df_test')]
test_files

['df_test_std_6.parquet',
 'df_test_std_5.parquet',
 'df_test_std_3.parquet',
 'df_test_std_0.parquet',
 'df_test_std_7.parquet',
 'df_test_std_2.parquet',
 'df_test_std_1.parquet',
 'df_test_std_4.parquet']

In [18]:
k = 0
for filename in train_files:
    if k == 0:
        df_train_std = pd.read_parquet(os.path.join(std_dir, filename))
        df_train_std  = reduce_mem_usage(df_train_std)
        k += 1
    else:
        df_part = pd.read_parquet(os.path.join(std_dir, filename))
        df_part = reduce_mem_usage(df_part)
        df_train_std = df_train_std.merge(df_part, how='inner', on='id')

Используемая память до преобразования 0.921 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.250 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.921 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.250 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.921 Gb


...

Используемая память после преобразования: 0.250 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.921 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.250 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.921 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.250 Gb
Количество используемой памяти уменьшено на 72.8%


In [20]:
targets = pd.read_csv('raw_data/target.csv')
targets.head()

,id,flag
0,0,0
1,1,0
2,2,0
3,3,0
4,4,0


In [21]:
targets.shape

(3000000, 2)

In [22]:
df_train_std = df_train_std.merge(targets, how='inner', on='id')

In [23]:
df_train_std.shape

(2400000, 388)

In [24]:
df_train_std.head()

,id,pre_pterm_14_std,pre_pterm_15_std,pre_pterm_16_std,pre_pterm_17_std,pre_fterm_1_std,pre_fterm_2_std,pre_fterm_3_std,pre_fterm_4_std,pre_fterm_5_std,...,pre_util_12_std,pre_util_13_std,pre_util_14_std,pre_util_15_std,pre_util_16_std,pre_util_17_std,pre_util_18_std,pre_util_19_std,pre_over2limit_1_std,flag
0,1805221,0.993652,-0.552246,-0.599121,-0.630371,-0.538574,-0.536621,-0.240112,-0.010277,0.712891,...,-0.289795,-0.279785,-0.061218,2.435547,0.539551,-0.090088,-0.437744,-0.282227,-0.0186,0
1,424193,0.993652,-0.552246,-0.599121,-0.630371,-0.538574,2.203125,0.591797,-0.010277,0.712891,...,-0.289795,-0.279785,-0.061218,-0.353516,1.310547,-0.090088,-0.437744,-0.282227,-0.0186,0
2,896388,0.179810,0.726562,1.860352,-0.630371,-0.538574,2.203125,0.591797,-0.010277,-0.479004,...,-0.289795,-0.279785,-0.061218,-0.353516,1.310547,-0.090088,-0.437744,-0.282227,-0.0186,0
3,2592966,0.179810,-0.552246,-0.599121,-0.630371,-0.538574,-0.536621,-0.240112,-0.010277,0.712891,...,-0.289795,-0.279785,-0.061218,-0.353516,-1.002930,-0.090088,-0.437744,-0.282227,-0.0186,0
4,2313279,0.179810,-0.552246,-0.599121,0.613281,2.136719,-0.536621,-0.240112,-0.010277,-0.479004,...,-0.289795,-0.279785,-0.061218,-0.353516,0.539551,-0.090088,-0.437744,-0.282227,-0.0186,0


In [25]:
df_train_std.to_parquet('data_train_std.pq')

In [26]:
del df_train_std
gc.collect()

0

In [27]:
k = 0
for filename in test_files:
    if k == 0:
        df_test_std = pd.read_parquet(os.path.join(std_dir, filename))
        df_test_std  = reduce_mem_usage(df_test_std)
        k += 1
    else:
        df_part = pd.read_parquet(os.path.join(std_dir, filename))
        df_part = reduce_mem_usage(df_part)
        df_test_std = df_test_std.merge(df_part, how='inner', on='id')

Используемая память до преобразования 0.230 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.063 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.230 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.063 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.230 Gb


...

Используемая память после преобразования: 0.063 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.230 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.063 Gb
Количество используемой памяти уменьшено на 72.8%
Используемая память до преобразования 0.230 Gb


  0%|          | 0/51 [00:00<?, ?it/s]

Используемая память после преобразования: 0.063 Gb
Количество используемой памяти уменьшено на 72.8%


In [28]:
df_test_std = df_test_std.merge(targets, how='inner', on='id')
df_test_std.shape

(600000, 388)

In [29]:
df_test_std.head()

,id,enc_paym_3_2_std,enc_paym_3_3_std,enc_paym_4_1_std,enc_paym_4_2_std,enc_paym_4_3_std,enc_paym_5_1_std,enc_paym_5_2_std,enc_paym_5_3_std,enc_paym_6_1_std,...,pre_util_12_std,pre_util_13_std,pre_util_14_std,pre_util_15_std,pre_util_16_std,pre_util_17_std,pre_util_18_std,pre_util_19_std,pre_over2limit_1_std,flag
0,1176617,-0.170776,-0.717773,-0.466064,-0.164062,-0.424805,-0.455566,-0.157837,-0.530762,-0.443115,...,-0.289795,-0.279785,-0.061218,-0.353516,-1.195312,-0.090088,-0.437744,-0.282227,-0.0186,0
1,2635727,-0.170776,-0.717773,-0.466064,-0.164062,-0.804688,-0.455566,-0.157837,-0.881348,-0.443115,...,-0.289795,-0.279785,-0.061218,-0.353516,-1.002930,-0.090088,-0.437744,-0.282227,-0.0186,0
2,1634546,-0.170776,-0.717773,-0.466064,-0.164062,-0.804688,-0.455566,-0.157837,-0.881348,-0.443115,...,6.511719,-0.279785,-0.061218,-0.353516,-0.617188,-0.090088,-0.437744,-0.282227,-0.0186,0
3,394967,-0.170776,-0.717773,-0.466064,-0.164062,-0.804688,-0.455566,-0.157837,-0.881348,-0.443115,...,-0.289795,-0.279785,-0.061218,-0.353516,-1.002930,-0.090088,-0.437744,-0.282227,-0.0186,0
4,2323499,-0.170776,0.532715,-0.466064,-0.164062,0.334717,-0.455566,-0.157837,0.170410,-0.443115,...,-0.289795,3.250000,-0.061218,-0.353516,-0.424316,-0.090088,-0.437744,-0.282227,-0.0186,0


In [30]:
df_test_std.to_parquet('data_test_std.pq')

In [31]:
del df_test_std
gc.collect()

0